# `melt_gamma_tuning_ssa`

> Generated from `config/ais_0.1_SSA.py` by `config/make_step_notebooks.py`. Edit the driver and regenerate; changes made here are overwritten.

Apply the validated SSA ocean-melt calibration (gamma_0=300, refit deltaT).

**Reads** (`load_model` calls in this step):
- `f'{model_dir}/AIS3_SSA_relaxed.nc'`

**Writes** (`save_model` calls in this step):
- `out_path = f"{model_dir}/AIS3_melt_final_ssa{SSA_CALIB}{os.environ.get('AIS3_MELT_TAG', '')}.nc"`

**Run as a batch job** (from `config/notebooks/`; resources copied from `config/launch_ais_0.1_SSA.pbs`):

```bash
qsub -q normal -l ncpus=8,mem=190GB,walltime=25:00:00 -v NB=ais_0.1_SSA/02_melt_gamma_tuning_ssa.ipynb launch_notebook.pbs
```

Add run options to the `-v` list, e.g. `NB=...,AIS3_SSA_CALIB=<value>`. The executed copy is written next to this notebook as `executed/<name>.<jobid>.ipynb`. Interactive use needs a Jupyter session with at least those resources.

**Run options**: set them in the next cell (interactive) or with `qsub -v` (batch).

Status, results and known failures: [`docs/run_guide.md`](../../../docs/run_guide.md) and the worklogs in `docs/`.

### What this step does
Builds `AIS3_melt_final_ssa.nc`, the input to the SSA historical and projection steps, from
the validated SSA melt calibration. There is no solve. It loads `AIS3_SSA_relaxed.nc` as the
model shell and sets ISMIP6 quadratic-local melt:
- IMBIE2 basins and Zhou thermal forcing
- `gamma_0=300` and the SSA track's 16 refit `deltaT` values, hard-coded in the step

### Before you run
- The step refuses to overwrite an existing output. Archive it (`mv -n`) or set
  `AIS3_MELT_TAG`.
- To redo the calibration rather than apply it, run the standalone chain (run guide §6):
  1. `melt_gamma_sweep_ssa.py`
  2. `melt_deltaT_sensitivity_test_ssa.py`
  3. `ssa_melt_deltaT_basin_refit.py`
  4. `ssa_melt_deltaT_refit_recalibrate.py`

  Then copy the new values into this step.

### What to check afterwards
- 30 thermal-forcing depth layers, and `gamma_0=300` in the saved file.

### Known pitfalls
- `AIS3_melt_gamma_tuning_ssa.nc` (`gamma_0=5537.7`) is the superseded coarse pick. Nothing
  should read it.
- At `gamma_0=300`, melt responds to ocean warming only about 4% as strongly as the ISMIP7
  ocean models (worklog §6.1). Projections use `gamma_0=15000` (`AIS3_PROJ_MELT=high`) as the
  high-melt member.

_Hand-written notes: `config/notebooks/notes/melt_gamma_tuning_ssa.md` (edit there, then regenerate)._

In [ ]:
import os

# Uncomment to set an option for an interactive run. Values passed with
# `qsub -v` already arrive in the environment. Defaults shown are the code's own.
# os.environ['AIS3_SSA_CALIB'] = ...   # code default: ''
# os.environ['AIS3_MELT_TAG'] = ...   # code default: ''

## Setup

The preamble of `ais_0.1_SSA.py`: imports, helpers, settings and the data catalog.

#### Imports

In [ ]:
"""ACCESS-AIS3 pure-SSA production pipeline (Budd/Weertman friction law, p=q=1 -- the best
validated RMSE anywhere in this project, 60.4 m/yr, vs. the Schoof-based HO track's current
94-99 m/yr after several chunked-restart iterations). See docs/inversion_worklog.md for the
full history of how p=q=1 was found, and the "in your opinion" exchange in this session's
transcript for why a dedicated SSA production track is worth having alongside the HO one:
SSA is dramatically cheaper and already converged/stable, at the cost of no vertical-shear
physics (so it can't do thermal-coupled work the way the HO track can).

Deliberately self-contained -- does NOT import or exec ais_0.1.py, and does NOT touch its
shared `steps`/`save` globals. This project hit two real race-condition incidents this
session from toggling those globals while a job was mid-flight; a fully separate file with
its own local `steps`/`save` sidesteps that risk entirely, at the cost of some code
duplication (the handful of helper functions and Budd-specific step logic below are copied
from ais_0.1.py, not imported, so the two files can diverge safely and be run concurrently).

Friction source: models/AIS3_ssa_friction_inv_reg_lcurve/run_001_10_100_0.0001/ -- the real,
production-canonical location the RMSE=60.4 p=q=1 result was salvaged into (see
config/finalize_p1q1_reg_lcurve.py) from the ad-hoc script that originally produced it
(p1q1_reg_sweep2.py, no longer present as a file, only as raw execution output). This
finalization step recovers the ACTUAL computed result rather than re-deriving it from
scratch, which would otherwise require regenerating AIS3_param.nc with Budd config (it
currently has Schoof baked in, for the HO track) and re-running two expensive inversion
stages. Run finalize_p1q1_reg_lcurve.py once before this script's first stage.

Stage status (updated 2026-09-02):
  1. ssa_inverted_solve_budd -- FULLY IMPLEMENTED AND RUN (RMSE=65.67 m/yr full-continental,
     the best full-continental result in the project). AIS3_SSA_inverted.nc.
  2. ssa_relaxation_budd -- FULLY IMPLEMENTED AND RUN (mean |dH|=32.90m, max |dH|=1505.39m
     over the 20yr relaxation). AIS3_SSA_relaxed.nc.
  3. melt_gamma_tuning_ssa -- builds AIS3_melt_final_ssa.nc directly from the validated
     calibration (gamma_0=300 + per-basin refit deltaT_basin, hard-coded; 2026-09-29). The
     values come from the standalone refit chain (ssa_melt_deltaT_basin_refit.py /
     ssa_melt_deltaT_refit_recalibrate.py). The existing AIS3_melt_final_ssa.nc was built by
     finalize_ssa_melt_calibration.py. The step's old coarse gamma_0 sweep (pick 5537.7,
     AIS3_melt_gamma_tuning_ssa.nc) was superseded by that refit and removed.
  4. historical_dhdt_tuning_ssa -- RUN AND VALIDATED. Full 1995-2019 (24yr) transient against
     AIS3_melt_final_ssa.nc, checked against MIPKIT's dhdt_cpom: grounded RMSE=1.402 m/yr,
     independently reconfirmed via check_historical_ssa.py. AIS3_historical_1995_2019_SSA.nc.
  5. projection_ssp_ssa -- IMPLEMENTED (2026-09-29): anomaly forcing on the calibrated
     baseline (TF0 = Zhou, SMB0 = RACMO 1995-2014), CTRL/SSP/attribution/melt variants chosen
     per job via qsub -v, self-chaining 50-yr chunks to 2300. See launch_projection_ssa.pbs and
     docs/projection_experiments.md.

Both new stages use the plain `cluster` (48 cores/190GB/normal) config, not hugemem -- the
SSA mesh is ~15x smaller than the HO track's (no vertical layers), and both stages use the
same synchronous submit-and-wait pattern (`waitonlock` minutes + `load_only=False` in one
call) already validated throughout this pipeline, avoiding the "never actually submits" bug
found and fixed in ais_0.1.py's own ho_relaxation/historical_dhdt_tuning on their first real
run.
"""
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import os
import glob
import sys
from pathlib import Path
from scipy.spatial import Delaunay

# Site settings (paths, NCI project, ISSM build): edit config/site.env, not this file. Found
# from this script's folder, or from the working directory upwards in a generated notebook.
_here = Path(globals().get('__file__', Path.cwd() / '_')).resolve().parent
sys.path[:0] = [str(p) for p in (_here, *_here.parents) if (p / 'ais3_site.py').exists()][:1]
import ais3_site as site

os.chdir(site.ROOT)
os.environ['ISSM_DIR'] = site.ISSM_DIR

model_dir = f'{site.ROOT}/models'
execution_dir = f'{site.ROOT}/execution_SSA'

# Melt calibration for the whole SSA chain (worklog 6.29-6.31). '' (default) = production: gamma_0=300
# with per-basin refit deltaT. '_hm<gamma_0>' = member hm<gamma_0> of models/melt_calibration_hm.json
# (history-matched calibration, melt_history_match.py + melt_calibration_hm_refit.py), e.g. _hm4000.
# The tag is appended to the melt file, the historical run, the projection initial state and baseline
# and every projection experiment name, so no production file is read as input or overwritten.
SSA_CALIB = os.environ.get('AIS3_SSA_CALIB', '')
assert SSA_CALIB == '' or SSA_CALIB.startswith('_hm'), f'AIS3_SSA_CALIB={SSA_CALIB!r}'
# '_hm<g>sem1' (member of models/melt_calibration_hm_sem1.json, melt_calibration_hm_refit_sem1.py): melt on partly
# floating elements, groundingline.melt_interpolation = SubelementMelt1 in the historical run and every projection
# chunk. Needs the ISSM build with the ismip6 IsAllFloating test removed (Element::Ismip6FloatingiceMeltingRate):
# AIS3_ISSM_DIR must hold the marker file SEM1_PATCH. (Unpatched, the ismip6 class sets no melt on partly floating
# elements, so SubelementMelt1 would silently change nothing.)
CALIB_SEM1 = SSA_CALIB.endswith('sem1')
SEM1_MARKER = f'{site.ISSM_DIR}/SEM1_PATCH'
if CALIB_SEM1:
    assert os.path.exists(SEM1_MARKER), f'AIS3_SSA_CALIB={SSA_CALIB} needs the patched ISSM build ({SEM1_MARKER} missing)'
# Friction-law variant for historical_dhdt_tuning_ssa (worklog 6.32): '' = production Budd p=q=1 with the
# prescribed (Ehrenfeucht) N; p<P>_fixedN / p<P>_flotN = exponent p=P (q=1) with the prescribed N
# (coupling 3) or N from flotation (coupling 2), C rescaled once at the start so the initial basal drag
# is unchanged (rescale_friction_drag_preserving). Appended as _<variant> to the historical run's name.
SSA_FRICTION = os.environ.get('AIS3_SSA_FRICTION', '')
assert SSA_FRICTION in ('', 'p1_flotN', 'p3_fixedN', 'p3_flotN', 'p5_flotN'), f'AIS3_SSA_FRICTION={SSA_FRICTION!r}'
FRIC_TAG = f'_{SSA_FRICTION}' if SSA_FRICTION else ''
# Time-varying ocean forcing for historical_dhdt_tuning_ssa (worklog 6.34): '' = the static Zhou climatology;
# main / cold / warm / vary = ISMIP7 contemporary (OCX) pathway, v2, applied as an anomaly on the calibration
# climatology: TF(t) = TF_Zhou + (OCX_<pathway>(t) - OCX_main(t)), annual columns at year + 0.5. After 1995
# 'main' equals the climatology (so it reproduces the static run); 'vary' carries the observed-like 16-yr
# thermocline oscillation in the Amundsen, Getz and Aurora basins. Appended as _ocx<pathway>.
SSA_OCEAN = os.environ.get('AIS3_SSA_OCEAN', '')
assert SSA_OCEAN in ('', 'main', 'cold', 'warm', 'vary'), f'AIS3_SSA_OCEAN={SSA_OCEAN!r}'
OCEAN_TAG = f'_ocx{SSA_OCEAN}' if SSA_OCEAN else ''
# Prescribed observed ice front for historical_dhdt_tuning_ssa (worklog 6.37): 'greene' = moving front
# (ismovingfront=1, zero calving rate and frontal melt) with spclevelset fixed at every vertex from the
# Greene et al. (2022) masks in MIPKIT (icemask_greene, 24 epochs 1997.75-2021.2). Only observed CHANGE is
# imposed: floating model ice is removed at an epoch if it is ice in the first (1997) mask and not ice at that
# epoch (re-advance allowed); model ice outside the 1997 mask is left alone (dataset mismatch, not change).
# AIS3_HIST_END (default 2019) shortens the run for smoke tests. Tags _frontgreene and _to<end>.
SSA_FRONT = os.environ.get('AIS3_SSA_FRONT', '')
assert SSA_FRONT in ('', 'greene'), f'AIS3_SSA_FRONT={SSA_FRONT!r}'
HIST_END = float(os.environ.get('AIS3_HIST_END', '2019'))
FRONT_TAG = (f'_front{SSA_FRONT}' if SSA_FRONT else '') + (f'_to{HIST_END:.0f}' if HIST_END != 2019 else '')

#### `set_greene_front()` — Moving front prescribed from the observed Greene masks (observed retreat of floating ice only)

In [ ]:
def set_greene_front(md, t_end):
    """Moving front prescribed from the observed Greene masks (observed retreat of floating ice only)."""
    mip = xr.open_dataset(f'{site.ISMIP_DIR}/ISMIP7/AIS/obs/mipkit/AntarcticaObsISMIP7-v1.2.nc')
    gm = mip['icemask_greene']
    gx, gy = mip['x'].values.astype(float), mip['y'].values.astype(float)
    mx = np.asarray(md.mesh.x, dtype=float).ravel()
    my = np.asarray(md.mesh.y, dtype=float).ravel()
    ix = np.clip(np.rint((mx - gx[0]) / (gx[1] - gx[0])).astype(int), 0, gx.size - 1)
    iy = np.clip(np.rint((my - gy[0]) / (gy[1] - gy[0])).astype(int), 0, gy.size - 1)
    tt = mip['greene_mask_time'].values
    tdec = np.array([t.astype('datetime64[D]').astype(object).year +
                     (t.astype('datetime64[D]').astype(object).timetuple().tm_yday - 1) / 365.25 for t in tt])
    il0 = np.asarray(md.mask.ice_levelset, dtype=float).ravel()
    ol0 = np.asarray(md.mask.ocean_levelset, dtype=float).ravel()
    cand = (il0 < 0) & (ol0 < 0)
    g_first = gm.isel(greene_mask_time=0).values[iy, ix] == 1
    cand &= g_first
    nv = md.mesh.numberofvertices
    use = [k for k in range(tdec.size) if tdec[k] <= t_end + 1.0]
    cols = [np.append(il0, float(md.timestepping.start_time))]
    for k in use:
        gone = cand & (gm.isel(greene_mask_time=k).values[iy, ix] == 0)
        cols.append(np.append(np.where(gone, 1.0, il0), tdec[k]))
        print(f"   Greene front {tdec[k]:.2f}: {int(gone.sum())} floating model vertices beyond the observed front", flush=True)
    md.levelset.spclevelset = np.stack(cols, axis=1)
    md.transient.ismovingfront = 1
    md.calving = pyissm.model.classes.calving.default(md.calving)
    md.calving.calvingrate = np.zeros(nv)
    md.frontalforcings = pyissm.model.classes.frontalforcings.default(md.frontalforcings)
    md.frontalforcings.meltingrate = np.zeros(nv)
    md.levelset.kill_icebergs = 1

#### `set_ocx_ocean_forcing()` — Replace the static TF columns with annual TF0 + (OCX pathway - OCX main) columns for `years`

In [ ]:
def set_ocx_ocean_forcing(md, pathway, years):
    """Replace the static TF columns with annual TF0 + (OCX pathway - OCX main) columns for `years`."""
    from scipy.interpolate import RegularGridInterpolator
    root = f'{site.ISMIP_DIR}/ISMIP7/AIS/OCX/ocean'
    dp = xr.open_dataset(f'{root}/{pathway}/v2/tf_AIS_OCX_ocean_{pathway}_v2_1950-2025.nc')['tf']
    dm = xr.open_dataset(f'{root}/main/v2/tf_AIS_OCX_ocean_main_v2_1950-2025.nc')['tf']
    gx, gy = dp['x'].values.astype(float), dp['y'].values.astype(float)
    assert np.allclose(np.abs(dp['z'].values), np.abs(np.asarray(md.basalforcings.tf_depths, dtype=float))), 'OCX depth levels differ from TF0'
    nv = md.mesh.numberofvertices
    pts = np.column_stack([np.asarray(md.mesh.y, dtype=float).ravel(), np.asarray(md.mesh.x, dtype=float).ravel()])
    tf0 = np.stack([np.asarray(t, dtype=float)[:nv, 0] for t in md.basalforcings.tf])     # (nz, nv)
    cols = np.empty((tf0.shape[0], nv + 1, len(years)))
    fl = np.asarray(md.mask.ocean_levelset).ravel() < 0
    for j, yr in enumerate(years):
        sp = dp.sel(time=dp['time.year'] == yr).isel(time=0).to_numpy()
        sm = dm.sel(time=dm['time.year'] == yr).isel(time=0).to_numpy()
        for k in range(tf0.shape[0]):
            a = RegularGridInterpolator((gy, gx), sp[k] - sm[k], method='linear', bounds_error=False, fill_value=np.nan)(pts)
            cols[k, :nv, j] = tf0[k] + np.nan_to_num(a, nan=0.0)
        cols[:, nv, j] = yr + 0.5
        d = cols[:, :nv, j] - tf0
        print(f"   OCX {pathway} {yr}: anomaly vs climatology over floating vertices, all depths: mean {d[:, fl].mean():+.3f} C, "
              f"min {d[:, fl].min():+.3f}, max {d[:, fl].max():+.3f}", flush=True)
    md.basalforcings.tf = [cols[k] for k in range(tf0.shape[0])]

#### `rescale_friction_drag_preserving()` — Budd p=q=1, coupling 3 -> exponent p (q=1), N prescribed or from flotation, same drag at u0

In [ ]:
def rescale_friction_drag_preserving(md, variant):
    """Budd p=q=1, coupling 3 -> exponent p (q=1), N prescribed or from flotation, same drag at u0.

    ISSM Budd drag: tau = C^2 N^(q/p) u^(1/p). Equal drag at the current speed u0 (m/s, floored at
    1 m/yr) gives C_new = C_old * sqrt(N_old * u0^(1 - 1/p) / N_new^(1/p)), as projection_ssp_ssa's
    coupledN / coupledN_p3. N_old is the prescribed N floored at limit*rho_i*g*H; N_new is that, or for
    *_flotN the flotation N (rho_i g H - rho_w g max(-base, 0), same floor) with coupling 2.
    """
    p_new = float(variant[1])
    assert int(md.friction.coupling) == 3, md.friction.coupling
    assert np.all(np.asarray(md.friction.p) == 1) and np.all(np.asarray(md.friction.q) == 1)
    g_, ri_, rw_ = float(md.constants.g), float(md.materials.rho_ice), float(md.materials.rho_water)
    H_ = np.asarray(md.geometry.thickness, dtype=float).ravel()
    base_ = np.asarray(md.geometry.base, dtype=float).ravel()
    floor_ = float(md.friction.effective_pressure_limit) * ri_ * g_ * H_
    N_old = np.maximum(np.maximum(np.asarray(md.friction.effective_pressure, dtype=float).ravel(), 0.0), floor_)
    N_new = np.maximum(ri_ * g_ * H_ - rw_ * g_ * np.maximum(-base_, 0.0), floor_) if variant.endswith('flotN') else N_old
    u0_ = np.maximum(np.asarray(md.initialization.vel, dtype=float).ravel(), 1.0) / float(md.constants.yts)
    ratio = np.sqrt(N_old * u0_ ** (1.0 - 1.0 / p_new) / np.maximum(N_new, 1e-6) ** (1.0 / p_new))
    md.friction.coefficient = np.asarray(md.friction.coefficient, dtype=float).ravel() * ratio
    md.friction.p = p_new * np.ones_like(np.asarray(md.friction.p, dtype=float))
    md.friction.q = np.ones_like(np.asarray(md.friction.q, dtype=float))
    if variant.endswith('flotN'):
        md.friction.coupling = 2
    gr_ = (np.asarray(md.mask.ocean_levelset).ravel() > 0) & (np.asarray(md.mask.ice_levelset).ravel() < 0)
    print(f"-- Friction {variant}: p={p_new:g}, coupling {int(md.friction.coupling)}; grounded N_new/N_old median "
          f"{np.median(N_new[gr_] / N_old[gr_]):.3f}; C factor median {np.median(ratio[gr_]):.3f} "
          f"(p5 {np.percentile(ratio[gr_], 5):.3f}, p95 {np.percentile(ratio[gr_], 95):.3f})")

#### Settings

In [ ]:
save = True
diagnostics = True
plot = True

cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR'] + '/bin'
cluster.executionpath = execution_dir
cluster.storage = site.STORAGE
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']
cluster.np = 48
cluster.memory = 190
cluster.time = 60 * 48
cluster.login = site.USER
cluster.project = site.PROJECT

FRICTION_RUN_DIR = f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/run_001_10_100_0.0001'
FRICTION_RUN_NAME = 'run_001_10_100_0.0001'

all_steps = ['ssa_inverted_solve_budd', 'ssa_relaxation_budd', 'melt_gamma_tuning_ssa', 'historical_dhdt_tuning_ssa', 'projection_ssp_ssa']
steps = ['historical_dhdt_tuning_ssa']  # edit before each qsub -- see launch_ais_0.1_SSA.pbs
# Per-job override via `qsub -v AIS3_STEPS=a,b` (same as ais_0.1.py) -- lets a launcher pick its
# step without editing the shared `steps` above.
if os.environ.get('AIS3_STEPS'):
    steps = os.environ['AIS3_STEPS'].split(',')

#### `_interp_weights()` — Precompute a reusable linear-interpolation transform (Delaunay triangulation +

In [ ]:
def _interp_weights(src_x, src_y, tgt_x, tgt_y):
    """Precompute a reusable linear-interpolation transform (Delaunay triangulation +
    barycentric weights) ONCE for a fixed (source-grid, target-mesh) pair, so many
    different value arrays defined on that same source grid can each be interpolated in
    milliseconds afterward via _interp_apply.

    Needed for projection_ssp_ssa's time-varying ocean thermal forcing: 30 depths x ~81
    years means ~2430 interpolations of the SAME 761x761 source grid onto the SAME mesh --
    pyissm.data.interp.points_to_mesh (scipy.interpolate.griddata under the hood) rebuilds
    the full Delaunay triangulation AND redoes point-location for every single call, which
    is fine for this pipeline's existing handful-of-calls loops (RACMO's 24 annual SMB
    calls) but empirically verified to cost ~13-20s PER CALL even with triangulation reuse
    alone (evaluation/point-location dominates, not triangulation) -- at 2430 calls that is
    hours just for TF preprocessing, dwarfing the actual transient solve. Precomputing the
    per-target-point simplex + barycentric weights ONCE instead drops each subsequent
    interpolation to ~10ms (verified empirically), since applying already-known weights to
    a new value array is just a weighted sum, not a fresh geometric search.
    """
    src_xy = np.column_stack([np.asarray(src_x).ravel(), np.asarray(src_y).ravel()])
    tgt_xy = np.column_stack([np.asarray(tgt_x).ravel(), np.asarray(tgt_y).ravel()])
    tri = Delaunay(src_xy)
    simplex = tri.find_simplex(tgt_xy)
    vertices = np.take(tri.simplices, simplex, axis=0)
    temp = np.take(tri.transform, simplex, axis=0)
    delta = tgt_xy - temp[:, 2]
    bary = np.einsum('njk,nk->nj', temp[:, :2, :], delta)
    wts = np.hstack((bary, 1 - bary.sum(axis=1, keepdims=True)))
    return vertices, wts, simplex

#### `_interp_apply()` — Apply a transform from _interp_weights to a new value array defined on the same

In [ ]:
def _interp_apply(values, vtx, wts, simplex, fill_value=np.nan):
    """Apply a transform from _interp_weights to a new value array defined on the same
    source grid it was built from -- same interpolation this pipeline uses everywhere else
    (linear, NaN outside the source grid's convex hull), just without repeating the
    triangulation/point-location work each time."""
    values = np.asarray(values).ravel()
    out = np.einsum('nj,nj->n', values[vtx], wts)
    out[simplex == -1] = fill_value
    return out

#### `load_shelf_rheology_B()` — Same as ais_0.1.py's helper -- copied, not imported, deliberately (see module docstring)

In [ ]:
def load_shelf_rheology_B():
    """Same as ais_0.1.py's helper -- copied, not imported, deliberately (see module docstring).
    Floating-ice rheology B from the shelf rheology inversion, independent of friction law:
    models/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17/ (the ais_0.1.py
    rheology_lcurve_run). Until 2026-11-16 this loaded execution_newB_rheology/run_001_1_10_1e-17,
    which finalize_newB_rheology.py saved there unchanged."""
    run = 'run_004_1_10_1e-17'
    mr = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/{run}/{run}.nc')
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## Step `melt_gamma_tuning_ssa`

In [ ]:
steps = ['melt_gamma_tuning_ssa']  # this notebook's step; overrides the driver default and AIS3_STEPS

**Section comment from the driver** (kept as written; it records history and may be out of date):

### Stage 3: Ocean basal melt -- apply the validated calibration, SSA mesh

Builds AIS3_melt_final_ssa.nc (historical_dhdt_tuning_ssa's input) from the validated ISMIP6 quadratic-local melt calibration: gamma_0 = 300 m/yr with per-basin refit deltaT_basin, official IMBIE2 basins, Zhou thermal-forcing climatology (worklog 4k, 4n). No solve: the melt parameterisation is evaluated inside the downstream transients.

The calibration SEARCH is not rerun here. It is a multi-round chain of standalone scripts (melt_gamma_sweep_ssa.py -> melt_deltaT_sensitivity_test_ssa.py -> ssa_melt_deltaT_basin_refit.py -> ssa_melt_deltaT_refit_recalibrate.py; state in models/deltaT_refit_state_ssa.json), and this step hard-codes its result. The J1-only gamma_0 sweep with fixed (published) deltaT_basin this step used to run was removed: its pick (5537.7) is superseded once deltaT_basin is refit per basin (J1 then ~0 for every candidate; J2 decides, worklog 4c).

Loads AIS3_SSA_relaxed.nc as the calibration did; its geometry is the inverted (unrelaxed) geometry, by design (worklog 4z correction). Thickness was already restored upstream in AIS3_SSA_inverted.nc (restore_floored_thickness_ssa.py, worklog 4k).

Refuses to overwrite an existing output: archive it first (mv -n) or set AIS3_MELT_TAG.

In [ ]:
print("-------------------------------------------------------------")
print(" OCEAN BASAL-MELT CALIBRATION, SSA (VALIDATED: gamma_0=300, REFIT deltaT)")
print("-------------------------------------------------------------")

from scipy.interpolate import RegularGridInterpolator, NearestNDInterpolator

ISMIP7_OCEAN = f'{site.ISMIP_DIR}/ISMIP7/AIS/parameterisations/ocean'
ZHOU_TF_FILE = (f'{site.ISMIP_DIR}/ISMIP7/AIS/obs/ocean/climatology/'
                 'zhou_annual_06_nov/tf/v3/tf_AIS_obs_ocean_climatology_zhou_annual_06_nov_v3_1972-2024.nc')
BASIN_FILE = f'{ISMIP7_OCEAN}/imbie2/basin_numbers_ismip8km_v2.nc'
TIME_SENTINEL = 1e9

# Validated values: candidate g1 of ssa_melt_deltaT_basin_refit.py (converged round 5, run
# AIS3_ssa_deltaT_refit_g1_r5), chosen by ssa_melt_deltaT_refit_recalibrate.py (job
# 179305728, J2 minimum). delta_t[k] belongs to IMBIE2 basin k+1. Close to, but not the
# same as, the HO track's refit values (different mesh).
MELT_GAMMA0 = 300.0
MELT_DELTA_T = np.array([
    1.8467609040725819, 2.7988123113211087, 2.2562599573273427, 4.599712312498511,
    8.762072283674417, 5.8733775833670565, 3.728768234826943, 0.8688693456218887,
    7.1878933110812575, 12.258715419441897, 5.2128687613028335, 4.89285368124807,
    2.3120433144628567, 3.4396433109318245, 1.2381000126240504, 2.2497361133862843])
if SSA_CALIB:
    import json
    _mem = None
    for _f in sorted(glob.glob(f'{model_dir}/melt_calibration_hm*.json')):  # hm.json, hm_sem1.json
        _members = json.load(open(_f))['members']
        if SSA_CALIB.lstrip('_') in _members:
            assert _mem is None, f'{SSA_CALIB} defined in more than one melt_calibration_hm*.json'
            _mem = _members[SSA_CALIB.lstrip('_')]
    assert _mem is not None, f'no member {SSA_CALIB.lstrip("_")} in models/melt_calibration_hm*.json'
    MELT_GAMMA0 = float(_mem['gamma_0'])
    MELT_DELTA_T = np.asarray(_mem['delta_t'], dtype=float)
    print(f"-- AIS3_SSA_CALIB={SSA_CALIB}: gamma_0={MELT_GAMMA0}, deltaT "
          f"[{MELT_DELTA_T.min():.3f}, {MELT_DELTA_T.max():.3f}] (models/melt_calibration_hm*.json)")

out_path = f"{model_dir}/AIS3_melt_final_ssa{SSA_CALIB}{os.environ.get('AIS3_MELT_TAG', '')}.nc"
if save and os.path.exists(out_path):
    raise FileExistsError(f"{out_path} exists -- archive it first (mv -n) or set AIS3_MELT_TAG")

#### Loading relaxed SSA model

In [ ]:
print("-- Loading relaxed SSA model...")
md = pyissm.model.io.load_model(f'{model_dir}/AIS3_SSA_relaxed.nc')
md.results.TransientSolution = []  # 400-step relaxation history, unused downstream (worklog 4g)

#### Configuring ISMIP6 basal melt parameterisation (IMBIE2 basins)

In [ ]:
print("-- Configuring ISMIP6 basal melt parameterisation (IMBIE2 basins)...")
md.basalforcings = pyissm.model.classes.basalforcings.ismip6(md.basalforcings)
ds_basin = xr.open_dataset(BASIN_FILE)
basin_grid = ds_basin['basinNumber'].values
bxx, byy = np.meshgrid(ds_basin['x'].values, ds_basin['y'].values)
num_basins = int(np.nanmax(basin_grid)) + 1
assert num_basins == MELT_DELTA_T.size, f"{num_basins} basins vs {MELT_DELTA_T.size} delta_t values"
# basin_id is per element: nearest basin at each element centroid (2D mesh, no projection).
elx = np.asarray(md.mesh.elements).astype(int) - 1
mesh_x = np.asarray(md.mesh.x).ravel()
mesh_y = np.asarray(md.mesh.y).ravel()
ecx = mesh_x[elx[:, :3]].mean(axis=1)
ecy = mesh_y[elx[:, :3]].mean(axis=1)
basin_lookup = NearestNDInterpolator(np.column_stack([bxx.ravel(), byy.ravel()]), basin_grid.ravel())
md.basalforcings.basin_id = (basin_lookup(np.column_stack([ecx, ecy])) + 1).astype(float)
md.basalforcings.num_basins = num_basins
md.basalforcings.islocal = 1
md.basalforcings.gamma_0 = MELT_GAMMA0
md.basalforcings.delta_t = MELT_DELTA_T.copy()

#### Interpolating Zhou ocean thermal-forcing climatology onto the mesh

In [ ]:
print("-- Interpolating Zhou ocean thermal-forcing climatology onto the mesh...")
ds_tf = xr.open_dataset(ZHOU_TF_FILE)
tfx, tfy, tfz = ds_tf['x'].values, ds_tf['y'].values, ds_tf['z'].values
tf_full = ds_tf['tf'].values  # (z, y, x)
tf_list = []
for k in range(tfz.size):
    vals = RegularGridInterpolator((tfy, tfx), tf_full[k], method='linear',
                                    bounds_error=False, fill_value=np.nan)(np.column_stack([mesh_y, mesh_x]))
    tf_list.append(np.append(np.nan_to_num(vals, nan=0.0), TIME_SENTINEL).reshape(-1, 1))
md.basalforcings.tf = tf_list
md.basalforcings.tf_depths = tfz.copy()
print(f"   {tfz.size} depth layers ({tfz.min():.0f} to {tfz.max():.0f} m)")

# Forward-run settings the downstream transients inherit (they reset the rest themselves).
md.inversion.iscontrol = 0
md.masstransport.spcthickness = np.full(md.mesh.numberofvertices, np.nan)
md.smb.mass_balance = np.zeros(md.mesh.numberofvertices)

if save:
    print(f"Saving to {out_path}")
    pyissm.model.io.save_model(md, out_path)